In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from statsmodels.api import OLS, add_constant

price_data = pd.read_csv("../data/prices.csv", index_col = 0, parse_dates = True).dropna()
price_data.head()

,AAPL,MSFT
Date,,
2020-01-02,72.468277,152.505661
2020-01-03,71.763733,150.606735
2020-01-06,72.335556,150.995987
2020-01-07,71.995338,149.619263
2020-01-08,73.153503,152.002457


In [3]:
# Estimate a rolling β, computer the spread and the rolling z-score, returns the time series of the strategy returns
def backtest_strategy(price_data, beta_window=252, z_window=60, entry_threshold=2):

    rolling_beta = []
    for i in range(beta_window, len(price_data)):
        y = price_data["AAPL"].iloc[i-beta_window:i]
        X = add_constant(price_data["MSFT"].iloc[i-beta_window:i])
        model = OLS(y, X).fit()
        rolling_beta.append(model.params["MSFT"])

    rolling_beta = pd.Series(
        rolling_beta,
        index=price_data.index[beta_window:]
    )

    aligned_prices = price_data.loc[rolling_beta.index]
    spread = aligned_prices["AAPL"] - rolling_beta * aligned_prices["MSFT"]

    rolling_mean = spread.rolling(z_window).mean()
    rolling_std = spread.rolling(z_window).std()

    zscore = (spread - rolling_mean) / rolling_std

    position = pd.Series(0, index=zscore.index)
    position[zscore > entry_threshold] = -1
    position[zscore < -entry_threshold] = 1
    position = position.ffill()

    spread_returns = spread.diff()
    strategy_returns = position.shift(1) * spread_returns
    strategy_returns = strategy_returns.fillna(0)

    return strategy_returns



In [21]:
# Testing how robust is our strategy (see notes)
thresholds = [1.5, 2, 2.5 ,3]
results = []

for t in thresholds:
    returns = backtest_strategy(price_data, entry_threshold = t)

    annual_return = returns.mean() * 252
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol

    results.append(sharpe)

pd.DataFrame({"Threshold" : thresholds, "Sharpe": results })

,Threshold,Sharpe
0,1.5,-3.267523
1,2.0,-1.979721
2,2.5,-1.197182
3,3.0,-0.296959


### Interpretation
We look for:
- Gradual decline
- No single "Magic Number"
- Stability around 2

This table shows stability, but the main issue is that the sharpe ratios are negative, meaning it will lead to a loss in the investment, which can be from a range of issues (ie not stationary enough, high transaction costs)

In [13]:
 # Test robustness by changing the Z-window
z_windows = [30,60,90,120]
results = []

for w in z_windows:
    returns = backtest_strategy(price_data, z_window = w)

    annual_return = returns.mean()
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol

    results.append(sharpe)

pd.DataFrame({ "Z Window" : z_windows, "Sharpe": results})

,Z Window,Sharpe
0,30,-0.002673
1,60,-0.007856
2,90,-0.014851
3,120,-0.007595


In [23]:
# Walk-forward validation
# Essentially instead of a single train/test split, we train on an early period, test on the next block, and roll forward.

dates = price_data.index
split_points = [
    "2021-01-01",
    "2022-01-01",
    "2023-01-01"
]

for split in split_points:
    train = price_data.loc[:split]
    test = price_data.loc[split:]
    
    returns = backtest_strategy(test)
    
    sharpe = (returns.mean()*252) / (returns.std()*np.sqrt(252))
    print(f"Start {split} Sharpe:", sharpe)


Start 2021-01-01 Sharpe: -1.2616076533168106
Start 2022-01-01 Sharpe: 1.2491254607270068
Start 2023-01-01 Sharpe: nan
